# Vector Fitting a Resonant Four-Port Network

This notebook stress-tests Circulax rational fitting with the resonant four-port `Agilent_E5071B.s4p` measurement used by the [scikit-rf spiky-response example](https://scikit-rf.readthedocs.io/en/latest/examples/vectorfitting/vectorfitting_ex3_Agilent_E5071B.html). The data contain many narrow resonances between 0.5 and 4.5 GHz.

We deliberately reserve every fifth frequency for validation. We compare automatic AAA pole discovery in Circulax with the manually sized 53rd-order scikit-rf vector fit, then apply Circulax's pre-simulation validation gate. This is a stress test rather than a promise that every fit should pass: a useful validator must reject an inaccurate or non-passive model.

In [ ]:
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import skrf
from skrf.vectorFitting import VectorFitting

from circulax.fitting import (
    FitValidationError,
    evaluate_sparameter_model,
    evaluate_surface,
    fit_with_delay,
    project_surface_passive,
    surface_from_fit,
    validate_surface_fit,
)

plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})

## Load the Touchstone data and make a hold-out set

The file is vendored with its upstream BSD-3-Clause license, so this notebook is deterministic and does not download data at runtime. The network uses a 75 Ω reference impedance.

In [ ]:
candidates = [
    Path("examples/fitting/data/Agilent_E5071B.s4p"),
    Path("data/Agilent_E5071B.s4p"),
]
data_path = next(path for path in candidates if path.exists())
network = skrf.Network(data_path)
z0 = float(np.real(network.z0[0, 0]))

sample_index = np.arange(len(network.f))
validation_mask = sample_index % 5 == 0
training_mask = ~validation_mask

frequencies_train = network.f[training_mask]
S_train = network.s[training_mask]
frequencies_validation = network.f[validation_mask]
S_validation = network.s[validation_mask]

print(network)
print(f"Training samples: {training_mask.sum()}, held-out samples: {validation_mask.sum()}, z0: {z0:g} ohm")

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(12, 9), sharex=True)
for row in range(4):
    for col in range(4):
        magnitude_db = 20 * np.log10(np.maximum(np.abs(network.s[:, row, col]), 1e-8))
        axes[row, col].plot(network.f / 1e9, magnitude_db, lw=1)
        axes[row, col].set_title(f"S{row + 1}{col + 1}")
        if row == 3:
            axes[row, col].set_xlabel("Frequency (GHz)")
        if col == 0:
            axes[row, col].set_ylabel("Magnitude (dB)")
fig.suptitle("Agilent E5071B measurement: all 16 responses")
fig.tight_layout()

## Circulax: automatic AAA poles, delay extraction, and Y fitting

`fit_with_delay` estimates a per-port delay, de-embeds it, converts S to Y, discovers poles with AAA, solves the shared-pole residue problem, and returns a state-space model. Passivity enforcement is intentionally deferred here so that validation can show the raw trade-off. The modest `delay_scale` avoids over-de-embedding a network whose resonances are not dominated by a single propagation delay.

In [ ]:
start = time.perf_counter()
ss_circulax, delay, fit_metadata = fit_with_delay(
    S_train,
    frequencies_train,
    z0=z0,
    tol=1e-6,
    delay_scale=0.1,
    enforce_passive=False,
    verbose=False,
)
circulax_seconds = time.perf_counter() - start

S_circulax_train = evaluate_sparameter_model(ss_circulax, frequencies_train, delay, z0)
S_circulax_validation = evaluate_sparameter_model(ss_circulax, frequencies_validation, delay, z0)
S_circulax_all = evaluate_sparameter_model(ss_circulax, network.f, delay, z0)

print(f"Pole count: {fit_metadata['pole_count']}")
print(f"Extracted delays (ps): {np.round(delay * 1e12, 3)}")
print(f"Elapsed: {circulax_seconds:.3f} s")

## scikit-rf reference fit

The upstream example recommends one real pole and 26 complex-conjugate poles, giving model order 53. Unlike Circulax's AAA path, this order is supplied by the user. Both implementations see exactly the same training samples.

In [ ]:
training_network = skrf.Network(
    frequency=skrf.Frequency.from_f(frequencies_train, unit="Hz"),
    s=S_train,
    z0=network.z0[training_mask],
    name="Agilent_E5071B_training",
)
vf_skrf = VectorFitting(training_network)
with warnings.catch_warnings(record=True) as skrf_warnings:
    warnings.simplefilter("always")
    start = time.perf_counter()
    vf_skrf.vector_fit(n_poles_real=1, n_poles_cmplx=26)
    skrf_seconds = time.perf_counter() - start

def evaluate_skrf(vf, frequencies, n_ports=4):
    result = np.empty((len(frequencies), n_ports, n_ports), dtype=complex)
    for row in range(n_ports):
        for col in range(n_ports):
            result[:, row, col] = vf.get_model_response(row, col, frequencies)
    return result

S_skrf_train = evaluate_skrf(vf_skrf, frequencies_train)
S_skrf_validation = evaluate_skrf(vf_skrf, frequencies_validation)
S_skrf_all = evaluate_skrf(vf_skrf, network.f)

print(f"Model order: {vf_skrf.get_model_order(vf_skrf.poles)}")
print(f"Passive over scikit-rf's assessment domain: {vf_skrf.is_passive()}")
print(f"Elapsed: {skrf_seconds:.3f} s")
for warning in skrf_warnings:
    print(f"scikit-rf warning: {warning.message}")

## Accuracy and generalization comparison

The complex RMSE treats real and imaginary error together. Maximum absolute error prevents a small number of missed narrow resonances from being hidden by the average. Model orders differ, so this is a workflow comparison—not a same-order solver benchmark.

In [ ]:
def error_metrics(prediction, target):
    error = np.abs(prediction - target)
    return np.sqrt(np.mean(error**2)), np.max(error)

comparison = {
    "Circulax AAA": (
        fit_metadata["pole_count"],
        *error_metrics(S_circulax_train, S_train),
        *error_metrics(S_circulax_validation, S_validation),
    ),
    "scikit-rf VF": (
        vf_skrf.get_model_order(vf_skrf.poles),
        *error_metrics(S_skrf_train, S_train),
        *error_metrics(S_skrf_validation, S_validation),
    ),
}
print(f"{'method':<18} {'order':>7} {'train RMSE':>13} {'train max':>13} {'holdout RMSE':>14} {'holdout max':>13}")
for method, values in comparison.items():
    order, train_rmse, train_max, holdout_rmse, holdout_max = values
    print(f"{method:<18} {order:7d} {train_rmse:13.4e} {train_max:13.4e} {holdout_rmse:14.4e} {holdout_max:13.4e}")

In [ ]:
selected = [(0, 0), (1, 0), (2, 2), (3, 0)]
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (row, col) in zip(axes.flat, selected, strict=True):
    ax.plot(network.f / 1e9, 20 * np.log10(np.maximum(np.abs(network.s[:, row, col]), 1e-8)), label="data")
    ax.plot(network.f / 1e9, 20 * np.log10(np.maximum(np.abs(S_circulax_all[:, row, col]), 1e-8)), label="Circulax")
    ax.plot(network.f / 1e9, 20 * np.log10(np.maximum(np.abs(S_skrf_all[:, row, col]), 1e-8)), "--", label="scikit-rf")
    ax.scatter(frequencies_validation / 1e9, 20 * np.log10(np.maximum(np.abs(S_validation[:, row, col]), 1e-8)), s=8, color="black", label="held out")
    ax.set_title(f"S{row + 1}{col + 1}")
    ax.set_ylabel("Magnitude (dB)")
    ax.set_xlabel("Frequency (GHz)")
axes[0, 0].legend(ncols=2)
fig.tight_layout()

## Pre-simulation validation

A fitted state-space model can be wrapped as a one-corner rational surface, allowing the same validation machinery to be used for scalar and surface fits. Passivity is assessed on a denser frequency grid than either the training or held-out samples. `raise_for_simulation()` is the final gate.

In [ ]:
feature = np.ones((1, 1))
omega_scale = 2 * np.pi * network.f.max()
surface = surface_from_fit(ss_circulax, delay, omega_scale, z0=z0)
passivity_frequencies = np.linspace(network.f.min(), network.f.max(), 801)

report = validate_surface_fit(
    surface,
    S_train[None, ...],
    feature,
    frequencies_train,
    validation_S=S_validation[None, ...],
    validation_features=feature,
    validation_freqs=frequencies_validation,
    passivity_features=feature,
    passivity_freqs=passivity_frequencies,
    simulation_frequency_range=(network.f.min(), network.f.max()),
)
print(report.summary())
try:
    report.raise_for_simulation()
except FitValidationError as error:
    print(f"\nSimulation blocked as intended:\n{error}")

## What if passivity is enforced anyway?

The prototype hard projection adds the smallest common diagonal shifts to the fitted D and E terms that make the model passive on the requested grid. It guarantees the sampled physical constraint, but it cannot repair a poor pole topology. The second report therefore checks accuracy again after projection.

In [ ]:
passive_surface, shifts = project_surface_passive(surface, feature, passivity_frequencies)
passive_report = validate_surface_fit(
    passive_surface,
    S_train[None, ...],
    feature,
    frequencies_train,
    validation_S=S_validation[None, ...],
    validation_features=feature,
    validation_freqs=frequencies_validation,
    passivity_features=feature,
    passivity_freqs=passivity_frequencies,
    simulation_frequency_range=(network.f.min(), network.f.max()),
)
print(f"D shift: {float(shifts.conductance):.3e}; E shift: {float(shifts.slope):.3e}")
print(passive_report.summary())

## Conclusion

This measurement is intentionally difficult. The manually sized scikit-rf model generalizes substantially better at much lower order, while its own global passivity assessment still reports a violation. Circulax's current automatic AAA/Y pipeline also fails the accuracy and passivity gates; hard passivity projection fixes the physical margin but not the approximation. The correct action is therefore **not to simulate either unqualified model**.

That result is useful: it shows that pole count, training error, passivity, and held-out error must be considered together. It also provides a concrete regression target for improving shared AAA pole selection on multiport responses.